In [ ]:
!pip install pyspark


In [1]:
!wget "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-01.parquet" -O yellow_tripdata_2025-01.parquet
!wget "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-02.parquet" -O yellow_tripdata_2025-02.parquet
!wget "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-03.parquet" -O yellow_tripdata_2025-03.parquet
!wget "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-04.parquet" -O yellow_tripdata_2025-04.parquet
!wget "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-05.parquet" -O yellow_tripdata_2025-05.parquet
!wget "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-06.parquet" -O yellow_tripdata_2025-06.parquet


--2025-11-21 19:01:55--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-01.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 65.8.245.171, 65.8.245.51, 65.8.245.50, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|65.8.245.171|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 59158238 (56M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2025-01.parquet’

yellow_tripdata_202 100%[===================>]  56.42M   155MB/s    in 0.4s    

2025-11-21 19:01:55 (155 MB/s) - ‘yellow_tripdata_2025-01.parquet’ saved [59158238/59158238]

--2025-11-21 19:01:55--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-02.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 65.8.245.171, 65.8.245.51, 65.8.245.50, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|65.8.245.171|:443... connected.
HTTP request sent, awai

In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession
    .builder
    .appName("nyc-taxi-hw")
    .getOrCreate()
)

paths = [
    "yellow_tripdata_2025-01.parquet",
    "yellow_tripdata_2025-02.parquet",
    "yellow_tripdata_2025-03.parquet",
    "yellow_tripdata_2025-04.parquet",
    "yellow_tripdata_2025-05.parquet",
    "yellow_tripdata_2025-06.parquet",
]

df = spark.read.parquet(*paths)
df.printSchema()
df.show(5)


root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)

+--------+--------------------+---------------------+---------------+------

In [3]:
from pyspark.sql.functions import col

df_clean = df.filter(
    (col("tpep_pickup_datetime") >= "2025-01-01") &
    (col("tpep_pickup_datetime") <  "2025-07-01") &
    (col("tpep_dropoff_datetime") >= "2025-01-01") &
    (col("tpep_dropoff_datetime") <  "2025-07-01") &
    (col("trip_distance") > 0) &
    (col("passenger_count") > 0)
)


In [8]:
from pyspark.sql.functions import hour

df_enriched = (
    df_clean
    .withColumn("pickup_hour",  hour(col("tpep_pickup_datetime")))
    .withColumn("dropoff_hour", hour(col("tpep_dropoff_datetime")))
)

df_final_cols = df_enriched.select(
    "tpep_pickup_datetime",   # время посадки
    "tpep_dropoff_datetime",  # время высадки
    "passenger_count",        # количество пассажиров
    "trip_distance",          # дистанция поездки
    "PULocationID",           # идентификатор зоны посадки
    "DOLocationID",           # идентификатор зоны высадки
    "total_amount",           # полная стоимость поездки
    "pickup_hour",            # час посадки
    "dropoff_hour"            # час высадки
)


In [5]:
!wget "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv" -O taxi_zone_lookup.csv

--2025-11-21 19:15:47--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 65.8.245.171, 65.8.245.178, 65.8.245.50, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|65.8.245.171|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0s      

2025-11-21 19:15:47 (219 MB/s) - ‘taxi_zone_lookup.csv’ saved [12331/12331]



In [6]:
zones_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("taxi_zone_lookup.csv")
)

zones_raw.printSchema()
zones_raw.show(5)


root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows



In [9]:
from pyspark.sql.functions import col

pu_zones = zones_raw.select(
    col("LocationID").alias("PULocationID"),
    col("Zone").alias("pickup_zone")
)

do_zones = zones_raw.select(
    col("LocationID").alias("DOLocationID"),
    col("Zone").alias("dropoff_zone")
)

df_joined = (
    df_final_cols
    .join(pu_zones, on="PULocationID", how="left")
    .join(do_zones, on="DOLocationID", how="left")
)


In [10]:
from pyspark.sql.functions import count

hourly_by_zone = (
    df_joined
    .groupBy("pickup_zone", "pickup_hour")
    .agg(count("*").alias("trips"))
)


In [11]:
from pyspark.sql.functions import avg

pivoted = (
    hourly_by_zone
    .groupBy("pickup_zone")
    .pivot("pickup_hour", list(range(24)))  # столбцы 0..23
    .agg(avg("trips"))
)


In [12]:
output_path = "nyc_taxi_hw_result_parquet"

(
    pivoted
    .write
    .mode("overwrite")  # режим перезаписи
    .parquet(output_path)
)
